# Python Data Model

## Core Idea

Python's "Pythonic" feel comes from consistency: instead of `collection.len()`, you write `len(collection)`.

This works because of the **Data Model** — a set of special methods\
(dunder methods, e.g. `__len__`, `__getitem__`) that the Python interpreter calls automatically behind built-in syntax.

`obj[key]` → interpreter calls `obj.__getitem__(key)`
`len(obj)` → interpreter calls `obj.__len__()`

By implementing these methods on your own classes, your objects gain access to Python's built-in syntax and standard library functions "for free" — without writing them yourself.

"Dunder" = "double underscore before and after" (e.g. dunder-getitem = `__getitem__`).

In [1]:
# The French Deck example

import collections

Card = collections.namedtuple('Card', ['rank', 'suit'])

class FrenchDeck:
    ranks = [str(n) for n in range(2, 11)] + list('JQKA')
    suits = 'spades diamonds clubs hearts'.split()

    def __init__(self):
        self._cards = [Card(rank, suit) for suit in self.suits
                                         for rank in self.ranks]

    def __len__(self):
        return len(self._cards)

    def __getitem__(self, position):
        return self._cards[position]

## What __len__ and __getitem__ unlock automatically

| Feature | Code | Why it works |
|---|---|---|
| `len()` | `len(deck)` | direct — calls `__len__` |
| Indexing | `deck[0]`, `deck[-1]` | direct — calls `__getitem__` |
| Slicing | `deck[:3]`, `deck[12::13]` | `__getitem__` delegates to `self._cards[position]`, 
and list slicing already works — so slicing "rides along" for free |
| Iteration | `for card in deck` | if there's no `__iter__`, Python falls back to calling 
`__getitem__(0)`, `__getitem__(1)`, ... until it hits an `IndexError` |
| `in` operator | `Card('Q','hearts') in deck` | falls back to a sequential scan using iteration, 
since there's no `__contains__` |
| `random.choice(deck)` | works with zero extra code | `random.choice` just needs indexing + `len()` |
| `reversed(deck)` | works with zero extra code | derived from the sequence protocol |
| `sorted(deck, key=...)` | works with zero extra code | works on any iterable |

**The key insight**: none of these (random.choice, reversed, sorted, in, 
iteration) were written by you. They all came from the standard library or 
built-in syntax recognizing that `FrenchDeck` "looks like" a sequence — 
because it implements the same two special methods a real sequence does.

## Why `len(x)` instead of `x.len()`

For **built-in types** (list, str, etc.), `len()` doesn't call a method at 
all — CPython stores the size directly in a C struct field, so reading it 
is instant (no method-call overhead). For **your own classes**, `len()` 
falls back to calling your `__len__()` method.

This is a deliberate practicality tradeoff (quoting the Zen of Python: 
"practicality beats purity") — not an inconsistency. `len` and `abs` behave 
more like built-in unary operators than methods.

In [ ]:
import math

class Vector:
    def __init__(self, x=0, y=0):
        self.x = x
        self.y = y

    def __repr__(self):
        return f'Vector({self.x!r}, {self.y!r})'

    def __abs__(self):
        return math.hypot(self.x, self.y)

    def __bool__(self):
        return bool(abs(self))

    def __add__(self, other):
        x = self.x + other.x
        y = self.y + other.y
        return Vector(x, y)

    def __mul__(self, scalar):
        return Vector(self.x * scalar, self.y * scalar)

## Key points on Vector

- `+` and `*` (via `__add__`, `__mul__`) always **return a new object** — 
they never modify `self` or `other` in place. This is the expected 
convention for infix operators (same "new object, not mutated" idea you 
already know from `list + [x]` vs `.append()`).
- `!r` inside the f-string (`{self.x!r}`) forces the `repr()` of that value — 
important because it distinguishes `Vector(1, 2)` from `Vector('1', '2')`. 
Worth using this in your own `__repr__` for Friday's exercise.
- **`__repr__` vs `__str__`**: `__repr__` is for developers/debugging — should 
look like valid code that could recreate the object (`Vector(3, 4)`). `__str__` 
is for end users — called by `print()` and `str()`. **If you only implement 
one, implement `__repr__`** — Python falls back to it for `__str__` if you 
don't define one separately.
- `__bool__`: controls what `bool(my_object)` returns (and therefore `if my_object:`). 
If you don't define `__bool__`, Python falls back to `__len__` (`0` → falsy, else truthy). 
If neither exists, all instances are truthy by default.

## Dunder cheat sheet (most relevant ones for now)

| Dunder | Triggered by | 
|---|---|
| `__init__` | Object creation, `MyClass(...)` |
| `__repr__` | `repr(obj)`, console display, debugging |
| `__str__` | `str(obj)`, `print(obj)` |
| `__len__` | `len(obj)` |
| `__getitem__` | `obj[key]`, indexing, slicing, and iteration fallback |
| `__contains__` | `x in obj` |
| `__iter__` | `for x in obj` (preferred over `__getitem__` fallback) |
| `__eq__` | `obj == other` |
| `__add__` | `obj + other` |
| `__mul__` | `obj * other` |
| `__bool__` | `bool(obj)`, `if obj:` |
| `__hash__` | `hash(obj)`, using obj as a dict key / set element |